> Menyiapkan Dataset

In [1]:
import numpy as np
import pandas as pd

np.random.seed(212)
n = 3000
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas12 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "waktu_tunggu_menit": np.random.randint(2, 25, size=n),
})

prob_positif = 1 / (1 + np.exp((data_tugas12["waktu_tunggu_menit"] - 12) / 4))
data_tugas12["ulasan_positif"] = (np.random.rand(n) < prob_positif).astype(int)

data_tugas12.to_csv("tugas12_ulasan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas12)} baris")
print("Distribusi label:")
print(data_tugas12["ulasan_positif"].value_counts())

Dataset berhasil dibuat: 3000 baris
Distribusi label:
ulasan_positif
0    1610
1    1390
Name: count, dtype: int64


> SparkSession dan CSV

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as spark_sum, round as spark_round

spark = SparkSession.builder \
    .appName("Tugas12-KopiNusantara") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas12_ulasan.csv", header=True, inferSchema=True)
print("Versi Spark:", spark.version)
df.printSchema()
df.show(5)
print("Jumlah baris:", df.count())
df.groupBy("ulasan_positif").count().orderBy("ulasan_positif").show()

26/11/26 21:37:50 WARN Utils: Your hostname, pathann resolves to a loopback address: 127.0.1.1; using 192.168.1.26 instead (on interface wlp8s0)
26/11/26 21:37:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 21:37:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Versi Spark: 3.5.9
root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- waktu_tunggu_menit: integer (nullable = true)
 |-- ulasan_positif: integer (nullable = true)

+--------------+----------+-----------+---+------------------+--------------+
|     nama_menu|    cabang|waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|
+--------------+----------+-----------+---+------------------+--------------+
|     Kopi Susu|Yogyakarta|       Sore|  5|                 5|             1|
|        Es Teh|Yogyakarta|       Pagi|  1|                21|             0|
|    Cappuccino|Yogyakarta|       Sore|  5|                15|             0|
|Kentang Goreng|  Semarang|      Malam|  2|                 5|             1|
|     Americano|Yogyakarta|      Malam|  2|                10|             0|
+--------------+----------+-----------+---+------------------+--------------+
only showing top 

>> A. ML Pipeline

In [10]:
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml import Pipeline

idx_menu = StringIndexer(inputCol="nama_menu", outputCol="nama_menu_idx")
idx_cabang = StringIndexer(inputCol="cabang", outputCol="cabang_idx")
idx_waktu = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx")

ohe = OneHotEncoder(
    inputCols=["nama_menu_idx", "cabang_idx", "waktu_pesan_idx"],
    outputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec"]
)

assembler = VectorAssembler(
    inputCols=["nama_menu_vec", "cabang_vec", "waktu_pesan_vec", "qty", "waktu_tunggu_menit"],
    outputCol="features"
)

lr = LogisticRegression(featuresCol="features", labelCol="ulasan_positif")

pipeline_lr = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, lr])

print("Pipeline Kopi Nusantara berhasil dibuat dengan", len(pipeline_lr.getStages()), "tahap.")

Pipeline Kopi Nusantara berhasil dibuat dengan 6 tahap.


>> B. Pelatihan dan Evaluasi

In [11]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator, BinaryClassificationEvaluator

train, test = df.randomSplit([0.8, 0.2], seed=77)
print(f"Data training: {train.count()} | Data testing: {test.count()}")

model_lr = pipeline_lr.fit(train)
pred_lr = model_lr.transform(test)

ev_acc = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="accuracy")
ev_f1 = MulticlassClassificationEvaluator(labelCol="ulasan_positif", predictionCol="prediction", metricName="f1")
ev_auc = BinaryClassificationEvaluator(labelCol="ulasan_positif", metricName="areaUnderROC")

acc_lr = ev_acc.evaluate(pred_lr)
f1_lr = ev_f1.evaluate(pred_lr)
auc_lr = ev_auc.evaluate(pred_lr)

print("\n=== Evaluasi Logistic Regression ===")
print(f"Accuracy : {acc_lr:.4f}")
print(f"F1-score : {f1_lr:.4f}")
print(f"AUC-ROC  : {auc_lr:.4f}")

print("\nConfusion Matrix Logistic Regression:")
pred_lr.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

Data training: 2411 | Data testing: 589

=== Evaluasi Logistic Regression ===
Accuracy : 0.7929
F1-score : 0.7926
AUC-ROC  : 0.8650

Confusion Matrix Logistic Regression:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  257|
|             0|       1.0|   57|
|             1|       0.0|   65|
|             1|       1.0|  210|
+--------------+----------+-----+



>> C. Random Forest

In [12]:
rf = RandomForestClassifier(featuresCol="features", labelCol="ulasan_positif", numTrees=30, seed=42)
pipeline_rf = Pipeline(stages=[idx_menu, idx_cabang, idx_waktu, ohe, assembler, rf])

model_rf = pipeline_rf.fit(train)
pred_rf = model_rf.transform(test)

acc_rf = ev_acc.evaluate(pred_rf)
f1_rf = ev_f1.evaluate(pred_rf)
auc_rf = ev_auc.evaluate(pred_rf)

print("\n=== Evaluasi Random Forest ===")
print(f"Accuracy : {acc_rf:.4f}")
print(f"F1-score : {f1_rf:.4f}")
print(f"AUC-ROC  : {auc_rf:.4f}")

print("\nConfusion Matrix Random Forest:")
pred_rf.groupBy("ulasan_positif", "prediction").count().orderBy("ulasan_positif", "prediction").show()

perbandingan = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest (30 pohon)"],
    "Accuracy": [acc_lr, acc_rf],
    "F1-score": [f1_lr, f1_rf],
    "AUC-ROC": [auc_lr, auc_rf]
}).round(4)
print("=== Perbandingan Model ===")
print(perbandingan.to_string(index=False))


=== Evaluasi Random Forest ===
Accuracy : 0.7963
F1-score : 0.7958
AUC-ROC  : 0.8474

Confusion Matrix Random Forest:
+--------------+----------+-----+
|ulasan_positif|prediction|count|
+--------------+----------+-----+
|             0|       0.0|  261|
|             0|       1.0|   53|
|             1|       0.0|   67|
|             1|       1.0|  208|
+--------------+----------+-----+

=== Perbandingan Model ===
                   Model  Accuracy  F1-score  AUC-ROC
     Logistic Regression    0.7929    0.7926   0.8650
Random Forest (30 pohon)    0.7963    0.7958   0.8474


>> D. Anlisis kesalahan

In [13]:
if auc_lr >= auc_rf:
    nama_terbaik, pred_terbaik = "Logistic Regression", pred_lr
else:
    nama_terbaik, pred_terbaik = "Random Forest", pred_rf
print(f"Model terbaik berdasarkan AUC: {nama_terbaik}\n")

salah = pred_terbaik.filter(col("ulasan_positif") != col("prediction"))
print("Jumlah prediksi salah:", salah.count(), "dari", pred_terbaik.count(), "data testing\n")

salah.select(
    "nama_menu", "cabang", "waktu_pesan", "qty", "waktu_tunggu_menit",
    "ulasan_positif", "prediction", "probability"
).show(10, truncate=False)

print("Tingkat kesalahan per waktu_tunggu_menit:")
pred_terbaik.groupBy("waktu_tunggu_menit").agg(
    count("*").alias("jumlah_data"),
    spark_sum(when(col("ulasan_positif") != col("prediction"), 1).otherwise(0)).alias("jumlah_salah")
).withColumn(
    "persen_salah", spark_round(col("jumlah_salah") / col("jumlah_data") * 100, 1)
).orderBy("waktu_tunggu_menit").show(30)

Model terbaik berdasarkan AUC: Logistic Regression

Jumlah prediksi salah: 122 dari 589 data testing

+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|nama_menu|cabang  |waktu_pesan|qty|waktu_tunggu_menit|ulasan_positif|prediction|probability                             |
+---------+--------+-----------+---+------------------+--------------+----------+----------------------------------------+
|Americano|Magelang|Malam      |1  |15                |1             |0.0       |[0.6253879175414343,0.3746120824585657] |
|Americano|Magelang|Siang      |2  |22                |1             |0.0       |[0.9113822566149014,0.08861774338509865]|
|Americano|Magelang|Sore       |1  |7                 |0             |1.0       |[0.23209137939008387,0.7679086206099162]|
|Americano|Magelang|Sore       |2  |11                |0             |1.0       |[0.42615454016905935,0.5738454598309406]|
|Americano|Magelang|Sore       |3  |5

Pada hasil di atas, kesalahan prediksi tidak tersebar merata. Tingkat kesalahan paling tinggi muncul pada waktu_tunggu_menit sekitar 10 sampai 14 menit, yaitu di sekitar titik tengah (12 menit) tempat probabilitas ulasan positif pada data berada di kisaran 50%.

In [14]:
atribut = pred_lr.schema["features"].metadata["ml_attr"]["attrs"]
semua_atribut = []
for tipe in atribut:
    semua_atribut.extend(atribut[tipe])
nama_fitur = [a["name"] for a in sorted(semua_atribut, key=lambda a: a["idx"])]

koef_lr = model_lr.stages[-1].coefficients.toArray()
imp_rf = model_rf.stages[-1].featureImportances.toArray()

tabel_fitur = pd.DataFrame({
    "fitur": nama_fitur,
    "koefisien_LR": koef_lr,
    "importance_RF": imp_rf
}).sort_values("importance_RF", ascending=False).round(4)

print(tabel_fitur.to_string(index=False))

                   fitur  koefisien_LR  importance_RF
      waktu_tunggu_menit       -0.2220         0.9430
                     qty       -0.0108         0.0113
 nama_menu_vec_Kopi Susu        0.1171         0.0094
   waktu_pesan_vec_Malam        0.2530         0.0059
   cabang_vec_Yogyakarta        0.0645         0.0058
     cabang_vec_Semarang       -0.1140         0.0049
    waktu_pesan_vec_Sore        0.1857         0.0043
    waktu_pesan_vec_Pagi       -0.0651         0.0041
nama_menu_vec_Roti Bakar       -0.1443         0.0041
 nama_menu_vec_Americano       -0.0855         0.0033
    nama_menu_vec_Es Teh       -0.0191         0.0024
nama_menu_vec_Cappuccino        0.0194         0.0015


>> E. Kesimpulan

Model yang dibangun menunjukkan bahwa waktu_tunggu_menit adalah faktor yang paling menentukan apakah pelanggan memberi ulasan positif. Pada Random Forest, fitur ini menyumbang hampir seluruh importance (sekitar 94%), dan pada Logistic Regression koefisiennya negatif, artinya makin lama menunggu makin kecil peluang ulasan positif. Fitur lain seperti menu, cabang, waktu pesan, dan qty pengaruhnya sangat kecil. Karena itu, rekomendasi pertama untuk manajemen Kopi Nusantara adalah menekan waktu tunggu, dengan target utama menjaganya di bawah sekitar 10 menit di semua cabang. Dari analisis kesalahan, rentang 10 sampai 14 menit adalah zona abu-abu di mana respons pelanggan paling sulit diprediksi, jadi pesanan yang mendekati 10 menit sebaiknya sudah diberi tindakan, misalnya diprioritaskan barista, atau pelanggan diberi informasi estimasi waktu dan permintaan maaf kecil seperti voucher kecil.

In [16]:
spark.stop()